
###### 24_nlp_rag

###### Purpose

This notebook is to combine Vector Search with a Large Language Model (LLM) to answer user questions using retrieved support ticket information.

###### Technologies Used

- Databricks

- Delta Lake

- Unity Catalog

- Databricks Vector Search

- Databricks Embedding Foundation Model (databricks-gte-large-en)

- LLM Model (databricks-meta-llama-3-1-8b-instruct)

###### Input

- Existing Vector Search Endpoint

- Existing Vector Search Index

- Delta table with precomputed embeddings

- User question

- Embedding Model

- LLM Model

- Prompt

######  Output

- LLM generated grounded response

######  Architecture

```text

User Question
       ↓
Embedding Model
      ↓
Question Embedding
       ↓
Vector Search Index
      ↓
Top-k Relevant Support Tickets
       ↓
Build Context
       ↓
Prompt Construction
       ↓
Large Language Model (LLM)
       ↓
Grounded Response
     
```


Section 1 - Import Libraries

In [0]:
# -----------------------------
# Install once if needed
# -----------------------------

%pip install databricks-vectorsearch
dbutils.library.restartPython()


In [0]:
# -----------------------------
# Import libraries
# -----------------------------
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
w = WorkspaceClient()

Section 2 - Configuration

In [0]:
# -----------------------------
# Define names
# -----------------------------

#source delta table
SOURCE_TABLE = "dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_embeddings"

#Vector Search endpoint name
VECTOR_SEARCH_ENDPOINT_NAME = "support-ticket-vector-search-endpoint"

#Vector index name
INDEX_NAME = (
    "dbw_agentic_ai_dev.support_ticket_ai."
    "gold_support_ticket_embeddings_index"
)

#Embedding model name
EMBEDDING_MODEL_NAME = "databricks-gte-large-en"

#LLM endpoint name
LLM_MODEL_NAME = "databricks-meta-llama-3-1-8b-instruct"

Section 3 - Connect to Vector Search Index

In [0]:
# -----------------------------------
# Connect to Vector Search
# -----------------------------------

vsc = VectorSearchClient()

index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=INDEX_NAME
)

Section 4 - Create Retrieval Function


In [0]:
def retrieve_documents(question, num_results=3):
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL_NAME,
        input=[question]
    )

    question_embedding = [
        float(x) for x in response.data[0].embedding
    ]

    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["ticket_id", "cleaned_ticket_text", "category"],
        num_results=num_results
    )

    return results["result"]["data_array"]





Section 5 - Build Context Function


In [0]:
def build_context(rows):
    context_lines = []

    for ticket_id, cleaned_ticket_text, category, score in rows:
        context_lines.append(
            f"Support Ticket {ticket_id}: {cleaned_ticket_text} | Category: {category}"
        )

    return "\n".join(context_lines)

Section 6 - Build Prompt Function


In [0]:
def build_prompt(question, context):
    prompt = f"""
You are a helpful telecom customer support assistant.

Use only the retrieved support ticket context below.

Your job:
1. Identify the most likely support ticket category.
2. Summarize what similar customers reported.
3. Provide a short helpful response.
4. If the user is asking for an action such as cancellation, login help, billing help, or technical support, respond with the likely support category and ask for any missing details needed to proceed.

Do not invent account-specific details, charges, dates, refunds, or actions that are not in the context.

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt

In [0]:
# System instruction,  Retrieved context,  User question
    
def build_prompt(question, context):
    prompt = f"""
You are a helpful telco customer support analyst.

Answer the user's question using only the context below.
If the context does not contain enough information, say:
"I do not have enough information from the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt


Section 7 - Call LLM Function

In [0]:
def generate_answer(prompt):
    response = w.serving_endpoints.query(
        name=LLM_MODEL_NAME,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=300,
        temperature=0.0
    )

    return response.choices[0].message.content

Section 8 - Create RAG Function


In [0]:
def rag_answer(question):
    # retrieve and  build context
    rows = retrieve_documents(
        question=question,
        num_results=3
    )

    context = build_context(rows)
    
    #build prompt
    prompt = build_prompt(
        question=question,
        context=context
    )
    
    #call LLM
    answer = generate_answer(prompt)

    return answer, context, rows

Section 9 - Test RAG

In [0]:
question = "Video streaming keeps buffering?"

answer, context, rows = rag_answer(question)

print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print(context)

print("\nANSWER:")
print(answer)

In [0]:
question = "Login page keeps failing?"

answer, context, rows = rag_answer(question)

print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print(context)

print("\nANSWER:")
print(answer)

In [0]:

question = "Need explanation for extra fees?"

answer, context, rows = rag_answer(question)

print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print(context)

print("\nANSWER:")
print(answer)

In [0]:

question = "Please disconnect my service?"

answer, context, rows = rag_answer(question)

print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print(context)

print("\nANSWER:")
print(answer)

In [0]:
question = "Please disconnect my service?"

answer, context, rows = rag_answer(question)

print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print(context)

print("\nANSWER:")
print(answer)

###### Notebook Summary

- Get Configurations for Vector Search endpoint name,  Vector index name,  Embedding model name and LLM endpoint name.

- Connect to Vector Search Index.

- Create Retrieval Function to Convert question to embedding, Query Vector Search and Return top-k support tickets.

- Build Context Function to Convert retrieved rows into context text.

- Build Prompt Function to System instruction, Retrieved context and User question.

- Create Function to Call LLM

- Create RAG Function to retrieve,  build context,  build prompt and call LLM

- Test RAG



###### Key Learnings

- RAG combines retrieval and generation.
- Vector Search retrieves relevant context before calling the LLM.
- Prompt quality affects RAG answer quality.
- Grounded answers should use retrieved context instead of relying only on the LLM’s general knowledge.

###### Notebook Conclusion

- In this notebook, we built a Retrieval-Augmented Generation workflow that retrieves relevant support ticket context using Vector Search and passes that context to an LLM through a prompt.

- This enables grounded answer generation using project-specific support ticket data instead of relying only on the LLM's general knowledge.

- This will be used in the next notebook to extend the RAG workflow into an Agentic AI application.

%md

###### Next Notebook

25_nlp_agentic_rag

The purpose of this notebook is to extend the RAG workflow into an Agentic AI application that can decide when to retrieve context and generate a response.